# 01. 첫 백테스트 — 보험주 vs 비보험주

**질문:** 보험업 지표/리스크 선별 사고방식이 종목 선택에 도움이 되는가?

| 그룹 | 가설 | 내용 |
|---|---|---|
| 보험주 | H1 | 합산비율 개선 보험사가 시장을 이긴다 → **FISIS 데이터 필요, 02번 노트북에서** |
| 보험주 | H2 | 저PBR·고ROE 보험사 3곳이 보험주 전체를 이긴다 |
| 비보험주 | H1 | 저PER·고ROE 10종목이 KOSPI를 이긴다 |
| 비보험주 | H2 | 변동성 큰 종목·적자 종목을 걸러내면 낙폭이 줄어든다 (언더라이팅식 선별) |

**방식:** 매월 1일·16일 근처 거래일(약 15일 간격)마다 데이터를 보고 종목 재선정 · 동일가중

**현실 보정:** 바뀐 종목 비율만큼 거래비용 차감 · 배당수익 근사 반영 · 거래대금 하위 종목 제외 · 주요 이벤트 경고

**한계(리포트에 꼭 적기):** 상장폐지 종목은 마지막 가격으로 고정(생존 편향) · EPS/BPS는 KRX가 그 날짜에 보여주는 값(공시 시차 근사) · 배당은 연 배당수익률을 기간에 나눠 더한 근사치


## 0. 설정 — `.env`에 KRX 아이디/비번이 있어야 데이터가 받아져요

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # .env 의 KRX_ID, KRX_PW 를 먼저 불러온 뒤 pykrx를 import

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pykrx import stock

START_YEAR, END_YEAR = 2016, 2025   # 백테스트 기간
REBAL = "semimonthly"               # "semimonthly"(15일) / "monthly" / "annual" 바꿔가며 비교해보기
COST = 0.003                        # 종목을 100% 교체할 때 드는 왕복 비용 0.3% (수수료+세금+슬리피지)
INCLUDE_DIV = True                  # 배당 근사 반영 (보험주는 배당이 커서 중요)
RF = 0.03                           # 무위험이자율 (샤프 비율용)
TOP_N_UNIVERSE = 100                # 비보험 유니버스: KOSPI 시총 상위 100
MIN_TRADING_VALUE = 1e9             # 거래대금 10억 미만 종목은 제외 (실제로 사고팔기 어려움)
EVENT_WINDOW_DAYS = 7               # 결정일 ±7일 안에 이벤트가 있으면 경고
SPLIT_YEAR = 2022                   # 이 해부터를 '검증 구간'으로 따로 평가 (과최적화 체크)
PPY = 24                            # 1년 = 평가 시점 24개 (반월 단위)
os.makedirs("data", exist_ok=True)

## 1. 데이터 받기 (처음 한 번은 20~30분 걸리고, 이후엔 data/ 폴더 캐시를 씀)

In [ ]:
def biz_day(yyyymmdd):
    """해당 날짜 이후 가장 가까운 거래일"""
    return stock.get_nearest_business_day_in_a_week(yyyymmdd, prev=False)

today = pd.Timestamp.today().strftime("%Y%m%d")
grid_raw = []
for m in pd.date_range(f"{START_YEAR-1}-01-01", f"{END_YEAR+1}-01-01", freq="MS"):
    grid_raw += [m, m + pd.Timedelta(days=15)]          # 매월 1일, 16일
grid_raw = [d for d in grid_raw if d <= pd.Timestamp(f"{END_YEAR+1}-01-01")]
dates = sorted({biz_day(d.strftime("%Y%m%d")) for d in grid_raw})
dates = [d for d in dates if d <= today]
print(len(dates), "개 평가 시점:", dates[0], "~", dates[-1])

In [ ]:
def load_cached(name, fn):
    path = f"data/{name}.pkl"
    if os.path.exists(path):
        return pd.read_pickle(path)
    df = fn()
    df.to_pickle(path)
    time.sleep(0.5)   # KRX 서버에 무리 주지 않기
    return df

closes, caps, tvals = {}, {}, {}
for i, d in enumerate(dates):
    df = load_cached(f"cap_{d}", lambda: stock.get_market_cap(d, market="KOSPI"))
    closes[d], caps[d], tvals[d] = df["종가"], df["시가총액"], df["거래대금"]
    if i % 24 == 0: print("시세", d)

close = pd.DataFrame(closes).T.replace(0, np.nan).ffill()   # 상장폐지/정지 → 마지막 가격 유지
cap = pd.DataFrame(caps).T
tval = pd.DataFrame(tvals).T

if REBAL == "semimonthly":
    reb_dates = [d for d in dates if d[:4] >= str(START_YEAR)]
elif REBAL == "monthly":
    reb_dates = [d for d in dates if d[:4] >= str(START_YEAR) and int(d[6:]) < 15]
else:
    reb_dates = [d for d in dates if d[:4] >= str(START_YEAR) and d[4:6] == "01" and int(d[6:]) < 15]
reb_dates = sorted(set(reb_dates) | {dates[-1]})   # 마지막 날은 평가용

fund = {}
for i, d in enumerate(reb_dates):
    fund[d] = load_cached(f"fund_{d}", lambda: stock.get_market_fundamental(d, market="KOSPI"))
    if i % 24 == 0: print("재무", d)

kospi = stock.get_index_ohlcv(dates[0], dates[-1], "1001")["종가"]
kospi.index = kospi.index.strftime("%Y%m%d")
kospi = kospi.reindex(dates).ffill()
print("리밸런싱 횟수:", len(reb_dates) - 1)

## 2. 보험주 유니버스 — 이름이 맞는지 꼭 눈으로 확인하기

In [ ]:
INSURERS = ["000810", "005830", "001450", "000370", "000400", "000540",   # 손보
            "032830", "088350", "085620", "082640",                       # 생보
            "003690"]                                                     # 재보험
for t in INSURERS:
    try:
        print(t, stock.get_market_ticker_name(t))
    except Exception as e:
        print(t, "확인 실패 →", e)

## 3. 가설별 종목 선택 규칙

In [ ]:
def roe_table(d):
    f = fund[d].copy()
    f = f[(f["PER"] > 0) & (f["BPS"] > 0)]      # 적자·자본잠식 제외
    f["ROE"] = f["EPS"] / f["BPS"]
    return f

def liquid(d, tickers):
    tv = tval.loc[d].reindex(tickers).fillna(0)
    return [t for t in tickers if tv[t] >= MIN_TRADING_VALUE]

def universe_big(d):
    """비보험 유니버스: 시총 상위 N (보험주 제외, 거래대금 필터)"""
    c = cap.loc[d].drop(INSURERS, errors="ignore").dropna()
    return liquid(d, list(c.sort_values(ascending=False).index[:TOP_N_UNIVERSE]))

def pick_nonins_h1(d, n=10):
    """저PER + 고ROE 순위 합산 상위 n"""
    f = roe_table(d).reindex(universe_big(d)).dropna(subset=["PER", "ROE"])
    score = f["PER"].rank() + f["ROE"].rank(ascending=False)
    return list(score.nsmallest(n).index)

def pick_nonins_h2(d):
    """언더라이팅식 선별: 적자 제외 + 최근 1년 변동성 상위 30% 제외, 나머지 동일가중"""
    uni = universe_big(d)
    ok = set(roe_table(d).index)
    i = dates.index(d)
    hist = close.iloc[max(0, i-PPY):i+1][uni].pct_change().dropna(how="all")
    vol = hist.std()
    keep = vol[vol <= vol.quantile(0.7)].index
    return [t for t in uni if t in ok and t in keep]

def pick_ins_all(d):
    return [t for t in INSURERS if t in close.columns and not np.isnan(close.loc[d, t])]

def pick_ins_h2(d, n=3):
    """보험주 중 저PBR + 고ROE 순위 합산 상위 n"""
    f = roe_table(d).reindex(pick_ins_all(d)).dropna(subset=["PBR", "ROE"])
    score = f["PBR"].rank() + f["ROE"].rank(ascending=False)
    return list(score.nsmallest(n).index)

STRATS = {"Ins_All": pick_ins_all, "Ins_H2": pick_ins_h2,
          "NonIns_Uni": universe_big, "NonIns_H1": pick_nonins_h1, "NonIns_H2": pick_nonins_h2}

## 4. 이벤트 경고 — 결정일 근처에 큰 이벤트가 있으면 표시

In [ ]:
events = pd.read_csv("../data_events/events.csv", parse_dates=["date"])

def events_near(d, window=EVENT_WINDOW_DAYS):
    t = pd.Timestamp(d)
    return events[(events["date"] - t).abs() <= pd.Timedelta(days=window)]

flags = {d: events_near(d) for d in reb_dates[:-1]}
n_flag = sum(len(v) > 0 for v in flags.values())
print(f"전체 결정일 {len(flags)}회 중 이벤트 근처: {n_flag}회\n")
for d, ev in flags.items():
    if len(ev):
        print("⚠️", d, " / ".join(f"{r.date:%m-%d} {r.event}" for r in ev.itertuples()))

# 앞으로 다가올 이벤트 (events.csv 에 미래 날짜를 추가해두면 여기서 알려줌)
upcoming = events[events["date"] >= pd.Timestamp.today()].sort_values("date").head(5)
print("\n📅 다가오는 이벤트:", "없음 — events.csv 에 추가하세요" if upcoming.empty else "")
if not upcoming.empty: print(upcoming.to_string(index=False))

## 5. 백테스트 엔진 — 결정일마다 종목 재선정, 바뀐 비율만큼 비용 차감

In [ ]:
def backtest(pick_fn):
    values, holdings, turnovers = [], {}, []
    v, prev = 1.0, set()
    for k, r in enumerate(reb_dates[:-1]):
        nxt = reb_dates[k+1]
        tickers = pick_fn(r) or list(prev)
        new = set(tickers)
        turnover = 1.0 if not prev else len(new - prev) / len(new)
        turnovers.append(turnover)
        v *= (1 - COST * turnover)
        holdings[r] = tickers
        prev = new

        period = [d for d in dates if r <= d <= nxt]
        rel = close.loc[period, tickers] / close.loc[r, tickers]
        path = rel.mean(axis=1)
        if INCLUDE_DIV:
            div = fund[r].reindex(tickers)["DIV"].fillna(0).mean() / 100
            path = path * (1 + div / PPY) ** np.arange(len(period))
        path = v * path
        values.append(path if not values else path.iloc[1:])
        v = path.iloc[-1]
    return pd.concat(values), holdings, np.mean(turnovers)

def stats(s):
    r = s.pct_change().dropna()
    yrs = len(r) / PPY
    cagr = (s.iloc[-1] / s.iloc[0]) ** (1/yrs) - 1
    vol = r.std() * np.sqrt(PPY)
    mdd = (s / s.cummax() - 1).min()
    return pd.Series({"CAGR": cagr, "Vol": vol, "Sharpe": (cagr - RF) / vol, "MDD": mdd})

results, picks, turn = {}, {}, {}
for name, fn in STRATS.items():
    results[name], picks[name], turn[name] = backtest(fn)
results["KOSPI"] = kospi.loc[results["Ins_All"].index] / kospi.loc[reb_dates[0]]
curve = pd.DataFrame(results)
curve.index = pd.to_datetime(curve.index)

def show(c):
    t = c.apply(stats).T
    return t.style.format("{:.1%}", subset=["CAGR", "Vol", "MDD"]).format("{:.2f}", subset=["Sharpe"])

print("평균 회전율(결정일마다 바뀐 종목 비율):", {k: f"{v:.0%}" for k, v in turn.items()})
show(curve)

## 6. 개발 구간 vs 검증 구간 — 두 구간 모두 좋아야 진짜 전략

In [ ]:
split = pd.Timestamp(f"{SPLIT_YEAR}-01-01")
print(f"개발 구간 (~{SPLIT_YEAR-1})"); display(show(curve[curve.index < split]))
print(f"검증 구간 ({SPLIT_YEAR}~)");   display(show(curve[curve.index >= split]))

## 7. 그래프 (빨간 점선 = 선거·정치, 회색 = 기타 이벤트) & 최근 보유종목

In [ ]:
ax = curve.plot(figsize=(12, 5), logy=True, title=f"Cumulative value (log) - rebalance: {REBAL}")
for r in events.itertuples():
    if curve.index[0] <= r.date <= curve.index[-1]:
        ax.axvline(r.date, color="red" if r.category in ("election", "politics") else "gray",
                   ls=":", alpha=.5, lw=1)
ax.set_xlabel(""); ax.grid(alpha=.3)
plt.show()

for name in ["Ins_H2", "NonIns_H1"]:
    last = list(picks[name])[-1]
    print(f"[{name}] {last} 보유:", ", ".join(stock.get_market_ticker_name(t) for t in picks[name][last]))

## 8. 이벤트 구간 성과 — 이벤트가 낀 15일 동안 전략별 수익률

In [ ]:
rows, idx = [], curve.index
for r in events.itertuples():
    after, before = idx[idx >= r.date], idx[idx < r.date]
    if len(after) and len(before):
        row = (curve.loc[after[0]] / curve.loc[before[-1]] - 1).to_dict()
        row.update(date=r.date.date(), event=r.event)
        rows.append(row)
ev_tbl = pd.DataFrame(rows).set_index(["date", "event"])
ev_tbl.style.format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-.1, vmax=.1)

## 9. 해석 메모 (experiments.md 에 옮겨 적기)

- Ins_H2 vs Ins_All : 일반 지표(PBR·ROE)로 보험주를 고르는 게 의미 있었나?
- NonIns_H1 vs KOSPI : 가치 전략이 시장을 이겼나?
- NonIns_H2 vs NonIns_Uni : **수익은 비슷한데 MDD가 줄었나?** ← 언더라이팅식 선별의 핵심 지표
- REBAL 을 semimonthly / monthly / annual 로 바꿔 돌려보기 → 자주 바꿀수록 비용이 수익을 얼마나 깎나?
- 검증 구간에서도 결과가 유지되나? (개발 구간만 좋으면 과최적화 의심)
- 이벤트 구간에서 어떤 전략이 잘 버텼나?

**다음:** 02번 노트북 — FISIS 합산비율 데이터로 보험주 H1
